## 7c. Flights — AeroDataBox via RapidAPI

Arrivals and departures at London Heathrow via the [AeroDataBox](https://rapidapi.com/aedbx-aedbx/api/aerodatabox/) flights endpoint. You'll need a free RapidAPI key — sign up [here](https://rapidapi.com/auth/sign-up) and subscribe to the Basic (free) tier. The free plan is limited to ~150 requests/month so be resourceful. Feel free to look for alternative endpoints, but settlement will be based on this data provider.

Two query styles are available (max 12h window each):
- **By relative time:** `offset_minutes` + `duration_minutes` relative to now
- **By time range:** explicit local times `fromLocal` / `toLocal` (format: `YYYY-MM-DDTHH:mm`)

The API also supports several boolean filters — check the [AeroDataBox docs](https://rapidapi.com/aedbx-aedbx/api/aerodatabox/playground/apiendpoint_3dbf8f9a-22de-4a99-8e7d-e542f6e63e4f) to learn what's available.

**Relevant for:** LHR_COUNT (total flights in 24h), LHR_INDEX (imbalance metric per 30-min interval).

In [1]:
import pandas as pd
import requests
import json
import time
from datetime import datetime, timedelta

AERODATABOX_KEY = "954aeb65f0mshc43dfde89d2698bp14c7ddjsnc2bec6059409"  # Replace with your key
AERODATABOX_HOST = "aerodatabox.p.rapidapi.com"
AIRPORT = "LHR"

def fetch_flights_broad_range(start_str, end_str):
    """Fetches flight data in 12-hour chunks to bypass API limits."""
    start_dt = datetime.strptime(start_str, "%Y-%m-%dT%H:%M")
    end_dt = datetime.strptime(end_str, "%Y-%m-%dT%H:%M")
    
    master_data = {"arrivals": [], "departures": []}
    current_dt = start_dt
    
    while current_dt < end_dt:
        chunk_end = min(current_dt + timedelta(hours=12), end_dt)
        from_str = current_dt.strftime("%Y-%m-%dT%H:%M")
        to_str = chunk_end.strftime("%Y-%m-%dT%H:%M")
        
        print(f"📡 Fetching: {from_str} to {to_str}...")
        
        url = f"https://{AERODATABOX_HOST}/flights/airports/iata/{AIRPORT}/{from_str}/{to_str}?direction=Both"
        resp = requests.get(url, headers={
            "x-rapidapi-host": AERODATABOX_HOST, 
            "x-rapidapi-key": AERODATABOX_KEY
        })
        
        if resp.status_code == 200:
            data = resp.json()
            master_data["arrivals"].extend(data.get('arrivals', []))
            master_data["departures"].extend(data.get('departures', []))
        else:
            print(f"❌ Error {resp.status_code}: {resp.text}")
            
        current_dt = chunk_end
        time.sleep(1.2) # Rate limit protection
        
    return master_data

In [ ]:
def process_and_filter_flights(data):
    # Competition constraints
    window_start = pd.to_datetime("2026-02-28 12:00:00").tz_localize("Europe/London")
    window_end = pd.to_datetime("2026-03-01 12:00:00").tz_localize("Europe/London")

    def filter_list(flight_list, time_key):
        extracted = []
        for flight in flight_list:
            # 1. isOperator Filter
            if flight.get('codeshareStatus') != 'IsOperator':
                continue
            if flight.get('status') == 'Canceled':
                continue
            
            # 2. Extract Times
            m = flight.get('movement', {})
            # Use Revised if available, else Scheduled
            raw_time = m.get('revisedTime', {}).get('local') or m.get('scheduledTime', {}).get('local')
            
            if raw_time:
                # Convert to TZ-aware timestamp
                ts = pd.to_datetime(raw_time)
                if ts.tz is None: ts = ts.tz_localize("Europe/London")

                # 3. Competition Window Filter (28 Feb 12pm - 1 Mar 12pm)
                if window_start <= ts <= window_end:
                    extracted.append({
                        f'final_{time_key}_time': ts,
                        'status': flight.get('status'),
                        'number': flight.get('number')
                    })
        return pd.DataFrame(extracted)

    df_arrivals = filter_list(data.get('arrivals', []), 'arrival')
    df_departures = filter_list(data.get('departures', []), 'departure')

    # Save to CSV for the Pricing Engine
    df_arrivals.to_csv('arrivals.csv', index=False)
    df_departures.to_csv('departures.csv', index=False)
    
    print(f"✅ Filtered Results: {len(df_arrivals)} Arrivals, {len(df_departures)} Departures")

# EXECUTION
raw_json = fetch_flights_broad_range("2026-02-28T00:00", "2026-03-02T00:00")

with open("flight_data_raw.json", "w", encoding="utf-8") as f:
    json.dump(raw_json, f, indent=4)

with open("flight_data_raw.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

process_and_filter_flights(data)

✅ Filtered Results: 589 Arrivals, 604 Departures
